In [2]:
import pandas as pd
import os

RAW_DATA_PATH = os.path.join("..", "data", "raw")

# Dicionário mapeando o nome da variável para o nome do arquivo físico
files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Carrega todos os DataFrames dinamicamente em um dicionário
dfs = {}
for name, file_name in files.items():
    file_path = os.path.join(RAW_DATA_PATH, file_name)
    dfs[name] = pd.read_csv(file_path)
    print(f"Tabela carregada: {name} | Linhas, Colunas: {dfs[name].shape}")

print("\nTodos os datasets foram carregados com sucesso no dicionário 'dfs'.")

Tabela carregada: orders | Linhas, Colunas: (99441, 8)
Tabela carregada: items | Linhas, Colunas: (112650, 7)
Tabela carregada: customers | Linhas, Colunas: (99441, 5)
Tabela carregada: products | Linhas, Colunas: (32951, 9)
Tabela carregada: sellers | Linhas, Colunas: (3095, 4)
Tabela carregada: reviews | Linhas, Colunas: (99224, 7)
Tabela carregada: payments | Linhas, Colunas: (103886, 5)
Tabela carregada: geolocation | Linhas, Colunas: (1000163, 5)
Tabela carregada: category_translation | Linhas, Colunas: (71, 2)

Todos os datasets foram carregados com sucesso no dicionário 'dfs'.


In [3]:
import pandas as pd

def assess_data_quality(df_dictionary: dict) -> pd.DataFrame:
    """
    Avalia métricas básicas de qualidade de dados para um dicionário de DataFrames.
    
    Itera sobre as tabelas fornecidas e retorna um DataFrame consolidado 
    contendo o volume de linhas, colunas, valores nulos e registros duplicados.
    """
    quality_metrics = []
    
    for name, df in df_dictionary.items():
        metrics = {
            "tabela": name,
            "total_linhas": df.shape[0],
            "total_colunas": df.shape[1],
            "total_nulos": df.isna().sum().sum(),
            "linhas_duplicadas": df.duplicated().sum()
        }
        quality_metrics.append(metrics)
        
    quality_df = pd.DataFrame(quality_metrics)
    
    # Ordena o resultado pelo volume de linhas em ordem decrescente para priorização
    return quality_df.sort_values(by="total_linhas", ascending=False).reset_index(drop=True)

# Gera e exibe o relatório consolidado de qualidade de dados
df_quality_report = assess_data_quality(dfs)
display(df_quality_report)

,tabela,total_linhas,total_colunas,total_nulos,linhas_duplicadas
0,geolocation,1000163,5,0,261831
1,items,112650,7,0,0
2,payments,103886,5,0,0
3,orders,99441,8,4908,0
4,customers,99441,5,0,0
5,reviews,99224,7,145903,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [5]:
import pandas as pd

# 1. Tratamento da tabela de Geolocalização (Remoção de Duplicatas)
print("--- Tratamento de Duplicatas ---")
linhas_antes_geo = dfs["geolocation"].shape[0]
dfs["geolocation"] = dfs["geolocation"].drop_duplicates()
linhas_depois_geo = dfs["geolocation"].shape[0]
print(f"Tabela 'geolocation': {linhas_antes_geo - linhas_depois_geo} linhas duplicadas removidas com sucesso.\n")

# 2. Investigação aprofundada dos nulos
def investigate_nulls(df: pd.DataFrame) -> pd.DataFrame:

    null_counts = df.isnull().sum()
    null_percent = (df.isnull().sum() / len(df)) * 100
    
    df_nulls = pd.DataFrame({
        'total_nulos': null_counts,
        'percentual_nulos': null_percent
    })
    
    # Retorna apenas colunas com nulos, ordenadas do maior para o menor
    return df_nulls[df_nulls['total_nulos'] > 0].sort_values(by='total_nulos', ascending=False)

print("--- Detalhamento de Nulos: Tabela 'reviews' ---")
display(investigate_nulls(dfs["reviews"]))

print("\n--- Detalhamento de Nulos: Tabela 'orders' ---")
display(investigate_nulls(dfs["orders"]))

--- Tratamento de Duplicatas ---
Tabela 'geolocation': 0 linhas duplicadas removidas com sucesso.

--- Detalhamento de Nulos: Tabela 'reviews' ---


,total_nulos,percentual_nulos
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532



--- Detalhamento de Nulos: Tabela 'orders' ---


,total_nulos,percentual_nulos
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899


In [6]:
# 1. Tratamento da tabela reviews (Imputação de valores padrão)
dfs["reviews"] = dfs["reviews"].fillna({
    "review_comment_title": "Sem título",
    "review_comment_message": "Sem mensagem"
})

print("--- Verificação Pós-Tratamento: Tabela 'reviews' ---")
print(f"Total de nulos restantes na tabela reviews: {dfs['reviews'].isnull().sum().sum()}")

# 2. Investigação da tabela orders (Validação de Regra de Negócio)
mask_null_delivery = dfs["orders"]["order_delivered_customer_date"].isnull()
status_of_undelivered = dfs["orders"][mask_null_delivery]["order_status"].value_counts()

print("\n--- Status dos pedidos sem data de entrega final ---")
print(status_of_undelivered)

--- Verificação Pós-Tratamento: Tabela 'reviews' ---
Total de nulos restantes na tabela reviews: 0

--- Status dos pedidos sem data de entrega final ---
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [7]:
# 1. Remoção da anomalia (pedidos 'delivered' sem data de entrega)
mask_anomaly = (dfs["orders"]["order_status"] == "delivered") & (dfs["orders"]["order_delivered_customer_date"].isnull())

linhas_antes = dfs["orders"].shape[0]
dfs["orders"] = dfs["orders"][~mask_anomaly]
linhas_depois = dfs["orders"].shape[0]

print(f"--- Limpeza de Anomalias ---")
print(f"Linhas inconsistentes removidas: {linhas_antes - linhas_depois}")

# 2. Conversão de Tipos para Datetime
date_columns = [
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]

for col in date_columns:
    dfs["orders"][col] = pd.to_datetime(dfs["orders"][col])

print("\n--- Tipagem atualizada da tabela 'orders' ---")
print(dfs["orders"][date_columns].dtypes)

--- Limpeza de Anomalias ---
Linhas inconsistentes removidas: 8

--- Tipagem atualizada da tabela 'orders' ---
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [ ]:
import os

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("--- Iniciando a exportação para a Camada Prata (Processed) ---")

# Itera sobre o dicionário e salva cada DataFrame como um novo CSV
for name, df in dfs.items():
    # Adiciona o prefixo 'cleaned_' para diferenciar dos arquivos originais
    file_name = f"cleaned_{name}.csv"
    file_path = os.path.join(PROCESSED_DATA_PATH, file_name)
    
    # index=False é crucial para não exportar o índice do Pandas como uma coluna extra
    df.to_csv(file_path, index=False)
    print(f"Arquivo salvo: {file_name}")

print("\n Pipeline de limpeza concluído. Checkpoint físico gerado com sucesso.")